# Synthetic Astronomical Transient Reports

This notebook follows the Eris solution contract:

- reads data from `./dataset/public/`
- performs a real local validation split for model selection
- trains fitted models end to end inside the notebook
- writes predictions to `./working/submission.csv`

## Problem Understanding

The challenge is a multi-output NLP classification task over short synthetic survey narratives.
The clean labels are almost directly verbalized in the text, while the scored derived labels depend on combinations of the clean labels plus distance and luminosity cues.
To keep the validation aligned with the benchmark difficulty, this notebook uses a compositional pair-heldout split over `(transient_class, host_environment)` combinations.


## Imports and Configuration

The pipeline uses only standard Kaggle-style Python libraries: `pandas`, `numpy`, `scikit-learn`, `scipy`, and `catboost`.

In [ ]:
from __future__ import annotations

import json
import random
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC


warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLEAN_LABELS = [
    "transient_class",
    "host_environment",
    "spectral_regime",
]

SCORED_LABELS = [
    "variability_pattern",
    "distance_bin",
    "energy_tier",
    "followup_protocol",
    "precursor_category",
]

SUBMISSION_COLUMNS = [
    "id",
    "transient_class",
    "host_environment",
    "spectral_regime",
    "variability_pattern",
    "distance_bin",
    "energy_tier",
    "followup_protocol",
    "precursor_category",
]

NUMERIC_FEATURES = [
    "z",
    "logL",
    "light_years",
    "years_after_bb",
    "years_ago",
    "universe_age",
    "z_missing",
    "logL_missing",
    "kw_nearby",
    "kw_distant",
    "kw_intermediate",
    "kw_early_universe",
    "kw_high_redshift",
    "kw_local_supercluster",
    "kw_cosmological_distance",
]

DISTANCE_FEATURES = CLEAN_LABELS + NUMERIC_FEATURES
ENERGY_FEATURES = CLEAN_LABELS + NUMERIC_FEATURES
FOLLOWUP_FEATURES = [
    "transient_class",
    "spectral_regime",
    "variability_pattern",
    "distance_bin",
]
PRECURSOR_FEATURES = [
    "transient_class",
    "host_environment",
]


def discover_data_paths() -> tuple[Path, Path, Path | None, Path]:
    base = Path("./dataset/public")
    train_path = base / "train.csv"
    test_path = base / "test.csv"
    sample_path = base / "sample_submission.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError("Expected train.csv and test.csv under ./dataset/public/.")

    output_dir = Path("./working")
    output_dir.mkdir(parents=True, exist_ok=True)
    return train_path, test_path, sample_path if sample_path.exists() else None, output_dir


def set_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def make_pair_holdout_split(
    df: pd.DataFrame,
    holdout_pairs: int = 18,
    seed: int = SEED,
) -> tuple[pd.Index, pd.Index]:
    rng = np.random.default_rng(seed)
    pair_df = (
        df[["transient_class", "host_environment"]]
        .drop_duplicates()
        .assign(pair=lambda x: x["transient_class"] + " || " + x["host_environment"])
        .reset_index(drop=True)
    )

    class_counts = pair_df["transient_class"].value_counts().to_dict()
    env_counts = pair_df["host_environment"].value_counts().to_dict()
    held_pairs: list[str] = []

    for idx in rng.permutation(len(pair_df)):
        row = pair_df.iloc[idx]
        transient_class = row["transient_class"]
        host_environment = row["host_environment"]
        pair = row["pair"]
        if class_counts[transient_class] > 1 and env_counts[host_environment] > 1:
            held_pairs.append(pair)
            class_counts[transient_class] -= 1
            env_counts[host_environment] -= 1
            if len(held_pairs) == holdout_pairs:
                break

    pair_series = df["transient_class"] + " || " + df["host_environment"]
    val_mask = pair_series.isin(held_pairs)
    train_idx = df.index[~val_mask]
    val_idx = df.index[val_mask]
    return train_idx, val_idx


def add_numeric_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    text = out["narrative"].fillna("")

    out["z"] = pd.to_numeric(
        text.str.extract(r"z\s*=\s*([0-9]+(?:\.[0-9]+)?)", expand=False),
        errors="coerce",
    )

    logl_patterns = [
        r"log\s*\(?\s*L\s*\)?\s*=\s*([0-9]+(?:\.[0-9]+)?)",
        r"log\s+Luminosity\s+of\s+([0-9]+(?:\.[0-9]+)?)",
        r"peak\s+log\s+Luminosity\s+of\s+([0-9]+(?:\.[0-9]+)?)",
    ]
    logl_values = []
    for pattern in logl_patterns:
        logl_values.append(pd.to_numeric(text.str.extract(pattern, expand=False, flags=re.IGNORECASE), errors="coerce"))
    logl_df = pd.concat(logl_values, axis=1)
    out["logL"] = logl_df.bfill(axis=1).iloc[:, 0]

    light_year_match = text.str.extract(
        r"([0-9]+(?:\.[0-9]+)?)\s*(million|billion)\s+light-?years",
        expand=True,
    )
    light_year_value = pd.to_numeric(light_year_match[0], errors="coerce")
    light_year_scale = light_year_match[1].map({"million": 1e6, "billion": 1e9})
    out["light_years"] = light_year_value * light_year_scale

    years_after_bb_match = text.str.extract(
        r"([0-9]+(?:\.[0-9]+)?)\s*(million|billion)\s+years?\s+(?:after|post)\s+(?:the\s+)?Big\s*Bang",
        expand=True,
    )
    years_after_bb_value = pd.to_numeric(years_after_bb_match[0], errors="coerce")
    years_after_bb_scale = years_after_bb_match[1].map({"million": 1e6, "billion": 1e9})
    out["years_after_bb"] = years_after_bb_value * years_after_bb_scale

    years_ago_match = text.str.extract(
        r"([0-9]+(?:\.[0-9]+)?)\s*(million|billion)\s+years?\s+ago",
        expand=True,
    )
    years_ago_value = pd.to_numeric(years_ago_match[0], errors="coerce")
    years_ago_scale = years_ago_match[1].map({"million": 1e6, "billion": 1e9})
    out["years_ago"] = years_ago_value * years_ago_scale

    universe_age_match = text.str.extract(
        r"universe\s+was\s+approximately\s+([0-9]+(?:\.[0-9]+)?)\s*(million|billion)\s+years?\s+old",
        expand=True,
    )
    universe_age_value = pd.to_numeric(universe_age_match[0], errors="coerce")
    universe_age_scale = universe_age_match[1].map({"million": 1e6, "billion": 1e9})
    out["universe_age"] = universe_age_value * universe_age_scale

    keyword_map = {
        "kw_nearby": "nearby",
        "kw_distant": "distant",
        "kw_intermediate": "intermediate",
        "kw_early_universe": "early universe",
        "kw_high_redshift": "high-redshift",
        "kw_local_supercluster": "local supercluster",
        "kw_cosmological_distance": "cosmological distance",
    }
    for column_name, keyword in keyword_map.items():
        out[column_name] = text.str.contains(keyword, case=False, regex=False).astype(int)

    out["z_missing"] = out["z"].isna().astype(int)
    out["logL_missing"] = out["logL"].isna().astype(int)
    return out


def augment_with_clean_tokens(df: pd.DataFrame) -> pd.Series:
    return (
        df["narrative"].fillna("")
        + " CLS_"
        + df["transient_class"].astype(str).str.replace(" ", "_", regex=False)
        + " ENV_"
        + df["host_environment"].astype(str).str.replace(" ", "_", regex=False)
        + " SPEC_"
        + df["spectral_regime"].astype(str).str.replace(" ", "_", regex=False)
    )


class TextVectorizer:
    def __init__(self) -> None:
        self.word = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=50_000,
            sublinear_tf=True,
            lowercase=True,
        )
        self.char = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=60_000,
            sublinear_tf=True,
            lowercase=True,
        )

    def fit_transform(self, text: pd.Series) -> sparse.csr_matrix:
        return sparse.hstack(
            [self.word.fit_transform(text), self.char.fit_transform(text)]
        ).tocsr()

    def transform(self, text: pd.Series) -> sparse.csr_matrix:
        return sparse.hstack(
            [self.word.transform(text), self.char.transform(text)]
        ).tocsr()


@dataclass
class PipelineOutputs:
    clean_predictions: pd.DataFrame
    submission_frame: pd.DataFrame


class SyntheticTransientPipeline:
    def __init__(self, seed: int = SEED) -> None:
        self.seed = seed
        self.clean_vectorizer = TextVectorizer()
        self.clean_models: dict[str, LinearSVC] = {}
        self.variability_vectorizer = TextVectorizer()
        self.variability_model: LinearSVC | None = None
        self.distance_model: CatBoostClassifier | None = None
        self.energy_model: CatBoostClassifier | None = None
        self.followup_model: CatBoostClassifier | None = None
        self.precursor_model: CatBoostClassifier | None = None

    def fit(self, train_df: pd.DataFrame) -> None:
        set_seeds(self.seed)
        train_df = train_df.copy()

        print(f"Fitting clean-label text models on {len(train_df)} rows...")

        clean_text_features = self.clean_vectorizer.fit_transform(train_df["narrative"])
        for label in CLEAN_LABELS:
            model = LinearSVC(C=0.8)
            model.fit(clean_text_features, train_df[label])
            self.clean_models[label] = model

        print("Fitting variability text model...")
        variability_text = augment_with_clean_tokens(train_df)
        variability_features = self.variability_vectorizer.fit_transform(variability_text)
        self.variability_model = LinearSVC(C=0.7)
        self.variability_model.fit(variability_features, train_df["variability_pattern"])

        print("Fitting distance model...")
        self.distance_model = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=260,
            depth=6,
            learning_rate=0.08,
            random_seed=self.seed,
            verbose=False,
        )
        self.distance_model.fit(
            train_df[DISTANCE_FEATURES],
            train_df["distance_bin"],
            cat_features=[0, 1, 2],
        )

        print("Fitting energy model...")
        self.energy_model = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=260,
            depth=6,
            learning_rate=0.08,
            random_seed=self.seed,
            verbose=False,
        )
        self.energy_model.fit(
            train_df[ENERGY_FEATURES],
            train_df["energy_tier"],
            cat_features=[0, 1, 2],
        )

        print("Fitting follow-up model...")
        self.followup_model = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=220,
            depth=5,
            learning_rate=0.08,
            random_seed=self.seed,
            verbose=False,
        )
        self.followup_model.fit(
            train_df[FOLLOWUP_FEATURES],
            train_df["followup_protocol"],
            cat_features=[0, 1, 2, 3],
        )

        print("Fitting precursor model...")
        self.precursor_model = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=220,
            depth=5,
            learning_rate=0.08,
            random_seed=self.seed,
            verbose=False,
        )
        self.precursor_model.fit(
            train_df[PRECURSOR_FEATURES],
            train_df["precursor_category"],
            cat_features=[0, 1],
        )

    def predict(self, df: pd.DataFrame) -> PipelineOutputs:
        if self.variability_model is None:
            raise RuntimeError("Pipeline must be fit before prediction.")

        df = df.copy()

        clean_features = self.clean_vectorizer.transform(df["narrative"])
        clean_predictions = pd.DataFrame(index=df.index)
        for label in CLEAN_LABELS:
            clean_predictions[label] = self.clean_models[label].predict(clean_features)

        variability_input = df.copy()
        for label in CLEAN_LABELS:
            variability_input[label] = clean_predictions[label]
        variability_features = self.variability_vectorizer.transform(
            augment_with_clean_tokens(variability_input)
        )
        variability_pred = pd.Series(
            self.variability_model.predict(variability_features),
            index=df.index,
            name="variability_pattern",
        )

        derived_input = pd.concat([clean_predictions, df[NUMERIC_FEATURES]], axis=1)
        distance_pred = pd.Series(
            self.distance_model.predict(derived_input[DISTANCE_FEATURES]).reshape(-1),
            index=df.index,
            name="distance_bin",
        )
        energy_pred = pd.Series(
            self.energy_model.predict(derived_input[ENERGY_FEATURES]).reshape(-1),
            index=df.index,
            name="energy_tier",
        )

        followup_input = pd.DataFrame(
            {
                "transient_class": clean_predictions["transient_class"],
                "spectral_regime": clean_predictions["spectral_regime"],
                "variability_pattern": variability_pred,
                "distance_bin": distance_pred,
            },
            index=df.index,
        )
        followup_pred = pd.Series(
            self.followup_model.predict(followup_input[FOLLOWUP_FEATURES]).reshape(-1),
            index=df.index,
            name="followup_protocol",
        )

        precursor_input = pd.DataFrame(
            {
                "transient_class": clean_predictions["transient_class"],
                "host_environment": clean_predictions["host_environment"],
            },
            index=df.index,
        )
        precursor_pred = pd.Series(
            self.precursor_model.predict(precursor_input[PRECURSOR_FEATURES]).reshape(-1),
            index=df.index,
            name="precursor_category",
        )

        submission_frame = pd.DataFrame(
            {
                "transient_class": clean_predictions["transient_class"],
                "host_environment": clean_predictions["host_environment"],
                "spectral_regime": clean_predictions["spectral_regime"],
                "variability_pattern": variability_pred,
                "distance_bin": distance_pred,
                "energy_tier": energy_pred,
                "followup_protocol": followup_pred,
                "precursor_category": precursor_pred,
            },
            index=df.index,
        )

        return PipelineOutputs(
            clean_predictions=clean_predictions,
            submission_frame=submission_frame,
        )


def evaluate_local_split(train_df: pd.DataFrame) -> dict[str, float]:
    train_idx, val_idx = make_pair_holdout_split(train_df, holdout_pairs=18, seed=SEED)
    train_part = train_df.loc[train_idx].reset_index(drop=True)
    val_part = train_df.loc[val_idx].reset_index(drop=True)

    print(
        f"Local validation split: {len(train_part)} train rows, "
        f"{len(val_part)} pair-heldout validation rows."
    )

    pipeline = SyntheticTransientPipeline(seed=SEED)
    pipeline.fit(train_part)
    val_predictions = pipeline.predict(val_part).submission_frame

    scores: dict[str, float] = {}
    for label in SCORED_LABELS:
        scores[label] = float(
            f1_score(
                val_part[label],
                val_predictions[label],
                average="macro",
                zero_division=0,
            )
        )
    scores["mean_scored_macro_f1"] = float(np.mean([scores[label] for label in SCORED_LABELS]))
    scores["validation_rows"] = float(len(val_part))
    return scores


def main() -> None:
    set_seeds(SEED)
    train_path, test_path, sample_path, output_dir = discover_data_paths()
    print(f"Using train file: {train_path}")
    print(f"Using test file: {test_path}")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print(f"Loaded train shape: {train_df.shape}")
    print(f"Loaded test shape: {test_df.shape}")
    train_df = add_numeric_features(train_df)
    test_df = add_numeric_features(test_df)

    local_scores = evaluate_local_split(train_df)
    print("Local pair-heldout validation:")
    for key, value in local_scores.items():
        print(f"  {key}: {value:.4f}")

    print("\nRefitting all models on the full training set...")
    pipeline = SyntheticTransientPipeline(seed=SEED)
    pipeline.fit(train_df)
    test_outputs = pipeline.predict(test_df)

    submission = pd.DataFrame({"id": test_df["id"]})
    for column in SUBMISSION_COLUMNS[1:]:
        submission[column] = test_outputs.submission_frame[column]

    if sample_path is not None:
        sample_columns = list(pd.read_csv(sample_path, nrows=1).columns)
        submission = submission[sample_columns]
    else:
        submission = submission[SUBMISSION_COLUMNS]

    submission_path = output_dir / "submission.csv"
    submission.to_csv(submission_path, index=False)

    metrics_path = output_dir / "local_validation_scores.json"
    metrics_path.write_text(json.dumps(local_scores, indent=2))

    print(f"\nSaved submission to: {submission_path}")
    print(f"Saved local validation scores to: {metrics_path}")

## File Discovery and Data Loading

The notebook reads `train.csv`, `test.csv`, and `sample_submission.csv` from `./dataset/public/` and writes outputs only to `./working/`.

## Target and Schema Inference

The required submission columns are inferred from the challenge schema and matched to the sample submission when available.

## Preprocessing

Feature engineering extracts numeric cues that appear in multiple narrative templates:

- explicit redshift and luminosity values
- light-year and cosmological-time phrases
- a few distance-related keyword flags

Text models use TF-IDF word and character n-grams, while the structured downstream heads use CatBoost.

## Local Validation Setup

A compositional validation split holds out 18 unseen `(transient_class, host_environment)` pairs to better approximate the benchmark's unseen-pair slice.

## Model Training

Training stages:

- `LinearSVC` clean-label heads for `transient_class`, `host_environment`, and `spectral_regime`
- a `LinearSVC` variability head conditioned on the predicted clean labels via appended text tokens
- `CatBoostClassifier` heads for `distance_bin`, `energy_tier`, `followup_protocol`, and `precursor_category`

## Validation Results, Full-Data Retraining, and Submission Creation

The final cell runs the full workflow: load data, engineer features, score the pair-heldout validation split, retrain on all rows, and write `./working/submission.csv`.

In [ ]:
main()